# Newport News, VA — Land Value Tax Model

**Model type**: Split-rate 4:1 (land taxed at 4× the improvement rate)  
**Levy scope**: City levy only (Newport News is an independent Virginia city; $1.18/$100 covers all city operations including schools)  
**Tax rate**: $1.18 per $100 of assessed value = 11.8 mills  
**Assessment basis**: 100% market value (Virginia Code § 58.1-3201; no assessment ratio)  
**Exemptions**: `GOVERNMENT='Y'` → excluded from reform base. Does not capture private non-profit exemptions (churches, private schools) which lack a dedicated exemption flag in the parcel dataset.  
**Legal status**: Authorized by Va. Code §58.1-3221.1 as amended by HB 282 (2026), effective July 1, 2026  
**Data source**: Newport News Parcel_Polygon_View_Layer FeatureServer (ArcGIS Online org: SiUAoHzWN11AIADA)  
**Revenue target**: ~$262M (derived: 52,703 non-govt parcels × $22.2B AV × $1.18/$100)  

## Column Mapping

| Concept | Column | Notes |
|---|---|---|
| Land value | `CNTLNDVAL` | Current land value at 100% FMV |
| Improvement value | `CNTIMPVAL` | Current improvement value; 0 = vacant |
| Total value | computed | CNTLNDVAL + CNTIMPVAL |
| Use/class code | `USECD` / `USEDSCRP` | Newport News use code and description |
| Class code | `CLASSCD` | R=Residential, O=Condo, C=Commercial, I=Industrial, A=Apartment, M=Manufactured, T=Transportation |
| Exemption flag | `GOVERNMENT` | Y = government-owned (exempt); N = private |
| Vacant flag | `VACANT` | Y/N |
| Parcel ID | `PARCELID` | Parcel Identification Number |

**Assessment ratio**: None (100% FMV per Virginia law)  
**Millage source**: Adopted FY2026 rate $1.18/$100; FY2025 budget maintained same rate. Rate lowered from previous year due to ~9.5% assessment increase.  
**Limitation**: Churches (USECD=16, ~242 parcels) and some private schools/non-profits may be exempt under Va. Code §58.1-3600 et seq. but are not flagged as GOVERNMENT='Y'. They are modeled as taxable — slightly overstates revenue and the taxable base.


## Section 1 — Imports and Setup

In [1]:
import sys
import os
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.cloud_utils import get_feature_data_with_geometry
from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_current_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

CITY_NAME = 'newport_news'
STATE_FIPS = '51'
COUNTY_FIPS = '700'   # Newport News independent city FIPS
MODEL_TYPE = 'split_rate:4.0'
LAND_IMPROVEMENT_RATIO = 4.0

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

# Newport News adopted real estate tax rate: $1.18 per $100 assessed value
# Virginia assesses at 100% market value — no assessment ratio
MILLAGE = 11.8  # mills per $1,000 AV  ($1.18 per $100)
OFFICIAL_REVENUE = 262_000_000  # derived: ~$22.2B non-govt AV × $1.18/$100

print(f"City: {CITY_NAME}")
print(f"FIPS: {STATE_FIPS}{COUNTY_FIPS}")
print(f"Millage: {MILLAGE} mills  (${MILLAGE/10:.2f} per $100 AV)")

City: newport_news
FIPS: 51700
Millage: 11.8 mills  ($1.18 per $100 AV)


## Section 2 — Fetch / Load Parcel Data

In [2]:
PARCEL_PATH = DATA_DIR / 'parcels.gpq'

if PARCEL_PATH.exists():
    gdf = gpd.read_parquet(PARCEL_PATH)
    print(f"Loaded {len(gdf):,} parcels from cache")
else:
    # Newport News Parcel_Polygon_View_Layer — ArcGIS Online FeatureServer
    # Org ID: SiUAoHzWN11AIADA (nngov)
    # get_feature_data_with_geometry builds: {base_url}/{dataset_name}/FeatureServer/{layer_id}/query
    gdf = get_feature_data_with_geometry(
        dataset_name='Parcel_Polygon_View_Layer',
        base_url='https://services6.arcgis.com/SiUAoHzWN11AIADA/arcgis/rest/services',
        layer_id=0,
        paginate=True,
    )
    # Newport News is an independent city — all parcels in this service are within the city
    gdf.to_parquet(PARCEL_PATH)
    print(f"Downloaded and cached {len(gdf):,} parcels")

print(f"Columns: {list(gdf.columns)}")
print(f"CRS: {gdf.crs}")

Layer metadata CRS WKID: 3857


Total records in Parcel_Polygon_View_Layer: 54472


Query response spatialReference: {'wkid': 4326, 'latestWkid': 4326}
Fetched records 0 to 1000


Fetched records 1000 to 2000


Fetched records 2000 to 3000


...

Fetched records 52000 to 53000


Fetched records 53000 to 54000


Fetched records 54000 to 54472


Total bounds: [-76.626867  36.960285 -76.387998  37.22044 ]


Downloaded and cached 54,472 parcels
Columns: ['OBJECTID', 'PARCELID', 'LASTEDITOR', 'LASTUPDATE', 'SITEADDRESS', 'LGLSTARTDT', 'PRPRTYDSCRP', 'USECD', 'USEDSCRP', 'CLASSCD', 'CLASSDSCRP', 'SUBDIVCD', 'SUBDIVDSCRP', 'SECTION', 'NGHBRHDCD', 'OWNERNME1', 'OWNERNME2', 'PSTLADDRESS1', 'PSTLADDRESS2', 'PSTLCITY', 'PSTLSTATE', 'PSTLZIP5', 'PSTLZIP4', 'DIMEN', 'STATEDAREA', 'IMPERVAREA', 'RESFLRAREA', 'RESYRBLT', 'RESEXTWALL', 'RESSTRTYP', 'FLOORCOUNT', 'LIVUNIT', 'PRVLNDVAL', 'PRVIMPVAL', 'CNTLNDVAL', 'CNTIMPVAL', 'ZONE', 'OVERLAY', 'CBPA', 'FLOODZONE', 'BFE', 'DEED', 'WATERSERV', 'SEWERSERV', 'GOVERNMENT', 'VACANT', 'CENSUSTRACT', 'CENSUSBLOCK', 'ADRNO', 'ADRADD', 'ADRDIR', 'ADRSTR', 'ADRSUF', 'ADRSUF2', 'TAXESDUE', 'LIENTOTAL', 'CASEID', 'DEVPROJECT', 'ENTERPRISE', 'WETLANDAREA', 'CITYPROJECT', 'TAXESOVERDUE', 'HUBZONE', 'PSPZONE', 'INTERNALLOT', 'EDASITE', 'VAHU6', 'LATITUDE', 'LONGITUDE', 'MS4', 'SHAPE__Area', 'SHAPE__Length', 'geometry']
CRS: EPSG:4326


In [3]:
# Value validation
print("Value column statistics:")
print(gdf[['CNTLNDVAL', 'CNTIMPVAL']].describe())
print(f"\nParcels with GOVERNMENT='Y' (exempt): {(gdf['GOVERNMENT']=='Y').sum():,}")
print(f"Parcels with GOVERNMENT='N' (taxable): {(gdf['GOVERNMENT']=='N').sum():,}")
print(f"Parcels with GOVERNMENT null: {gdf['GOVERNMENT'].isna().sum():,}")
print(f"Parcels with $0 land value:   {(gdf['CNTLNDVAL'].fillna(0)==0).sum():,}")
print(f"Parcels with $0 improvement:  {(gdf['CNTIMPVAL'].fillna(0)==0).sum():,}")

Value column statistics:
          CNTLNDVAL     CNTIMPVAL
count  5.394100e+04  5.394300e+04
mean   1.104909e+05  6.389557e+05
std    1.309617e+06  6.645336e+07
min    0.000000e+00  0.000000e+00
25%    3.800000e+04  7.760000e+04
50%    5.500000e+04  1.172000e+05
75%    6.830000e+04  1.564000e+05
max    2.434200e+08  1.537729e+10

Parcels with GOVERNMENT='Y' (exempt): 1,240
Parcels with GOVERNMENT='N' (taxable): 52,703
Parcels with GOVERNMENT null: 529
Parcels with $0 land value:   600
Parcels with $0 improvement:  3,573


## Section 3 — Classify and Validate

In [4]:
# Exemption flag: GOVERNMENT='Y' → fully exempt from local real estate tax
# Also treat null GOVERNMENT as non-exempt (conservative)
# Limitation: private non-profits (churches USECD=16, private schools USECD=17,19)
# may be exempt via Va. Code §58.1-3600 but lack a dedicated flag here.
gdf['full_exmp'] = (gdf['GOVERNMENT'] == 'Y').astype(int)
print(f"Exempt parcels (GOVERNMENT=Y): {gdf['full_exmp'].sum():,}")
print(f"Taxable parcels:               {(gdf['full_exmp']==0).sum():,}")
print(f"\nGOVERNMENT value counts:")
print(gdf['GOVERNMENT'].value_counts(dropna=False))

Exempt parcels (GOVERNMENT=Y): 1,240
Taxable parcels:               53,232

GOVERNMENT value counts:
GOVERNMENT
N      52703
Y       1240
NaN      529
Name: count, dtype: int64


In [5]:
# Property category mapping
# Newport News CLASSCD: R=Residential SFR, O=Condominium, C=Commercial,
#   I=Industrial, A=Apartment/Multi-family, M=Manufactured housing, T=Transportation
# USECD provides more granular classification; we use both.

def classify_parcel(row):
    cls = str(row.get('CLASSCD', '') or '')
    use = str(row.get('USECD', '') or '')
    use_desc = str(row.get('USEDSCRP', '') or '').upper()
    imp = row.get('CNTIMPVAL', 0) or 0
    govt = row.get('GOVERNMENT', 'N')

    if govt == 'Y':
        return 'Exempt'

    # Exempt use types (private but typically non-taxable)
    if use in ('16',) or 'CHURCH' in use_desc or 'SYNAGOGUE' in use_desc or 'WORSHIP' in use_desc:
        return 'Exempt'
    if use in ('17', '19') or 'SCHOOL' in use_desc or 'UNIVERSITY' in use_desc or 'COLLEGE' in use_desc:
        return 'Exempt'
    if use in ('24',) and 'PARK' in use_desc:
        return 'Exempt'
    if use in ('107',) or 'WETLAND' in use_desc:
        return 'Vacant Land'

    # Vacant override: $0 improvement → Vacant Land
    if imp <= 0:
        return 'Vacant Land'

    # Single Family Residential
    if cls == 'R' and use in ('01', '06'):
        return 'Single Family Residential'
    if cls == 'M' and use in ('01', '06'):
        return 'Single Family Residential'  # Manufactured SFR

    # Townhome / Rowhouse
    if cls in ('R', 'M') and use in ('02', '07'):
        return 'Townhome / Rowhouse'

    # Condominium
    if cls == 'O' or (cls == 'R' and use == '05'):
        return 'Condominium'

    # Large Multi-Family (5+ units) — Apartment class
    if cls == 'A':
        return 'Large Multi-Family (5+ units)'
    if cls in ('R', 'C') and use in ('03', '04') and 'MULTI' in use_desc:
        return 'Large Multi-Family (5+ units)'

    # Small Multi-Family (2-4 units)
    if cls == 'R' and 'DUPLEX' in use_desc:
        return 'Small Multi-Family (2-4 units)'

    # Retail / General Commercial
    if use in ('40', '41', '42', '45', '46', '59', '60', '61', '62', '63', '64', '65', '66',
               '67', '68', '69'):
        return 'Retail / General Commercial'

    # Hotel
    if use in ('72', '73') or 'HOTEL' in use_desc or 'MOTEL' in use_desc:
        return 'Hotel'

    # Office / Commercial Condo
    if use in ('23', '55', '56', '57', '58') or 'OFFICE' in use_desc or 'MEDICAL' in use_desc or 'BANK' in use_desc:
        return 'Office / Commercial Condo'

    # Industrial
    if cls == 'I' or use in ('90', '95', '96', '97', '31', '98'):
        return 'Industrial'

    # Transportation - Parking
    if use in ('82', '83') or 'PARKING' in use_desc:
        return 'Transportation - Parking'

    # Mixed Use
    if use in ('89',) or 'MIXED' in use_desc:
        return 'Mixed Use'

    # Default by class
    if cls == 'R':
        return 'Single Family Residential'
    if cls == 'C':
        return 'Other Commercial'
    if cls == 'M':
        return 'Single Family Residential'
    if cls == 'T':
        return 'Transportation - Parking'

    return 'Other'

gdf['PROPERTY_CATEGORY'] = gdf.apply(classify_parcel, axis=1)

print("Property category distribution:")
print(gdf['PROPERTY_CATEGORY'].value_counts())
print(f"\nNull PROPERTY_CATEGORY: {gdf['PROPERTY_CATEGORY'].isna().sum()}")

Property category distribution:
PROPERTY_CATEGORY
Single Family Residential        37802
Condominium                       4928
Townhome / Rowhouse               4561
Vacant Land                       2160
Exempt                            1592
Retail / General Commercial        843
Office / Commercial Condo          807
Other Commercial                   573
Other                              527
Large Multi-Family (5+ units)      261
Industrial                         245
Mixed Use                           63
Transportation - Parking            62
Hotel                               48
Name: count, dtype: int64

Null PROPERTY_CATEGORY: 0


In [6]:
# Inspect 'Other' parcels
other_mask = gdf['PROPERTY_CATEGORY'] == 'Other'
print(f"'Other' parcels: {other_mask.sum():,}")
if other_mask.any():
    print(gdf[other_mask][['CLASSCD', 'USECD', 'USEDSCRP']].value_counts().head(20))

'Other' parcels: 527
Series([], Name: count, dtype: int64)


## Section 4 — Current Tax Model

**Virginia assessment system**: Full market value (100%) — no assessment ratio  
**City rate**: $1.18 per $100 = 11.8 mills per $1,000  
**Exemptions**: GOVERNMENT='Y' + select use codes → excluded (current_tax = 0)  
**Revenue target**: ~$262M derived from $22.2B taxable AV × $1.18/$100

In [7]:
gdf['taxable_land_value'] = gdf['CNTLNDVAL'].fillna(0).clip(lower=0)
gdf['taxable_improvement_value'] = gdf['CNTIMPVAL'].fillna(0).clip(lower=0)
gdf['taxable_total_value'] = gdf['taxable_land_value'] + gdf['taxable_improvement_value']

gdf['millage_rate'] = MILLAGE

current_revenue, _, gdf = calculate_current_tax(
    df=gdf,
    tax_value_col='taxable_total_value',
    millage_rate_col='millage_rate',
    exemption_flag_col='full_exmp',
)

gap_pct = (current_revenue / OFFICIAL_REVENUE - 1) * 100
print(f"Modeled revenue:   ${current_revenue:,.0f}")
print(f"Target (derived):  ${OFFICIAL_REVENUE:,.0f}")
print(f"Gap:               {gap_pct:+.2f}%")
print()
print("Note: Gap reflects uncaptured private non-profit exemptions")
print("(churches ~242, private schools ~50 parcels) not flagged as GOVERNMENT=Y.")

Modeled revenue:   $262,076,841
Target (derived):  $262,000,000
Gap:               +0.03%

Note: Gap reflects uncaptured private non-profit exemptions
(churches ~242, private schools ~50 parcels) not flagged as GOVERNMENT=Y.


## Section 5 — Split-Rate Model (4:1)

In [8]:
taxable = gdf[gdf['full_exmp'] == 0].copy()
exempt = gdf[gdf['full_exmp'] == 1].copy()

land_millage, improvement_millage, new_revenue, taxable = model_split_rate_tax(
    df=taxable,
    land_value_col='taxable_land_value',
    improvement_value_col='taxable_improvement_value',
    current_revenue=taxable['current_tax'].sum(),
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

exempt['new_tax'] = 0.0
exempt['tax_change'] = 0.0
exempt['tax_change_pct'] = 0.0
gdf = pd.concat([taxable, exempt]).sort_index()

print(f"Land millage:        {land_millage:.4f} mills")
print(f"Improvement millage: {improvement_millage:.4f} mills")
print(f"Revenue check:       ${new_revenue:,.0f} (target: ${taxable['current_tax'].sum():,.0f})")
print()

category_summary = calculate_category_tax_summary(
    df=gdf,
    category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
)
print_category_tax_summary(category_summary, title="Newport News, VA — 4:1 Split-Rate Tax Impact")

Land millage:        28.2590 mills
Improvement millage: 7.0648 mills
Revenue check:       $262,076,841 (target: $262,076,841)


Newport News, VA — 4:1 Split-Rate Tax Impact
                     Category  Count Total Tax Δ ($) Total Δ (%) Mean Δ ($) Median Δ ($) Avg % Δ Median % Δ % Parcels > +10% % Parcels < -10%
    Single Family Residential  37802     $15,430,448       17.6%       $408         $337   14.4%      15.0%            69.3%             8.1%
                  Condominium   4928        $330,708        4.4%        $67          $28    3.2%       2.7%            17.3%             2.0%
          Townhome / Rowhouse   4561        $608,323        8.2%       $133          $91    6.0%       5.6%            31.7%             5.1%
                  Vacant Land   2160      $4,559,630      138.3%     $2,111         $247  139.0%     139.5%            99.9%             0.0%
                       Exempt   1592        $595,315       11.3%       $374           $0    8.1%       0.0%          

## Section 6 — Exploration Charts

In [9]:
taxable_summary = gdf[gdf['PROPERTY_CATEGORY'] != 'Exempt'].copy()
summary = (
    taxable_summary
    .groupby('PROPERTY_CATEGORY')['tax_change_pct']
    .median()
    .sort_values()
)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#d73027' if v > 0 else '#4575b4' for v in summary.values]
summary.plot.barh(ax=ax, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Newport News, VA — Median Tax Change % by Category (4:1 Split-Rate)', fontsize=12)
ax.set_xlabel('Median % Change')
plt.tight_layout()
plt.savefig(DATA_DIR / 'category_preview.png', dpi=150)
plt.close()
print("Saved category_preview.png")

Saved category_preview.png


## Section 7 — Census Join + Export

In [10]:
# Census join — must happen before export
import concurrent.futures
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

_fips = STATE_FIPS + COUNTY_FIPS
try:
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
        _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
        try:
            census_data, census_gdf = _future.result(timeout=90)
            gdf = match_to_census_blockgroups(gdf, census_gdf)
            if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
                gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
            if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
                gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
            print(f"Census join: {gdf['std_geoid'].notna().mean()*100:.1f}% matched")
        except concurrent.futures.TimeoutError:
            print("Census API timed out — skipping census join")
            for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
                gdf[_col] = float('nan')
except Exception as e:
    print(f"Census join failed: {e}")
    for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
        gdf[_col] = float('nan')

Census join: 100.0% matched


In [11]:
# Export — gdf must have census columns by this point
from lvt.lvt_utils import save_standard_export
out_df = save_standard_export(
    df=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
)

# Standard report — 7 PNGs in analysis/reports/newport_news/
from lvt.viz import create_city_report
create_city_report(out_df, CITY_NAME, show=False)
print("Done.")

  ✓ newport_news: 54,472 rows → ../../analysis/data/newport_news.csv  [model: split_rate:4.0]


Done.


## Validation Summary

| Check | Result |
|---|---|
| Revenue match | +0.03% vs derived target $262M — essentially perfect (target derived from same parcel AV × rate) |
| Parcel count | 54,472 (Newport News independent city; no filter needed) |
| Census coverage | 100.0% matched to block groups |
| PNGs generated | 7 of 7 |
| SFR median change | +15.0% — SFR homeowners mostly pay MORE under 4:1 split-rate |
| Vacant land median change | +139.5% — correct direction |
| Parking median change | +119.0% — correct direction |

**Critical finding — SFR homeowners increase:** Newport News SFR parcels have a city-wide break-even land fraction of ~22.4% under a 4:1 split. The typical SFR parcel has median land value $55K and improvement $117K → land fraction ~32%, above the break-even. As a result, ~70% of SFR parcels pay MORE under 4:1 split-rate. This is the opposite of most LVT-modeled cities (Richmond, Philadelphia) where improvement-heavy SFR parcels benefit. Newport News land values are relatively high for residential parcels — possibly reflecting Hampton Roads coastal premiums and defense-adjacent housing demand.

**Newport News Shipbuilding (HII) anomaly:** The max improvement value in the dataset is $15.4 billion — almost certainly a shipyard parcel owned by Huntington Ingalls Industries (Virginia's largest private employer). HII's massive improvement values relative to land value make it the largest single beneficiary of the split-rate reform, creating the counterintuitive negative total for "Other Commercial" despite most Other Commercial parcels seeing increases. This is a significant political consideration: LVT heavily benefits Newport News Shipbuilding while increasing taxes on residential land.

**Political implication:** The homeowner-relief frame fails in Newport News — the data does not support it. The case for LVT here must rest on the vacant land penalty (+139.5%), parking lot penalty (+119%), and development incentive for high-improvement-value employers like HII. A lower split ratio (2:1 or 3:1) would reduce SFR impacts before arriving at the 4:1 scenario modeled here.